In [1]:
import pandas as pd


In [2]:
# ---- Bank Statement (Excel) ----
print("=" * 80)
print("BANK STATEMENT (SpendWise2k26.xlsx)")
print("=" * 80)
bk = pd.read_excel("CSVS/SpendWise2k26.xlsx")
print(f"Shape: {bk.shape}")
print(f"\nColumns: {list(bk.columns)}")
print(f"\nDtypes:\n{bk.dtypes}")
print(f"\nFirst 5 rows:")
print(bk.head().to_string())
print(f"\nLast 5 rows:")
print(bk.tail().to_string())
print(f"\nNull counts:\n{bk.isnull().sum()}")



BANK STATEMENT (SpendWise2k26.xlsx)
Shape: (1653, 12)

Columns: ['Transaction_Date', 'Debit', 'Credit', 'Balance', 'Transaction_Mode', 'DR/CR_Indicator', 'Transaction_ID', 'Recipient_Name', 'Bank', 'UPI_ID', 'Note', 'Amount']

Dtypes:
Transaction_Date     object
Debit               float64
Credit              float64
Balance             float64
Transaction_Mode     object
DR/CR_Indicator      object
Transaction_ID       object
Recipient_Name       object
Bank                 object
UPI_ID               object
Note                 object
Amount              float64
dtype: object

First 5 rows:
  Transaction_Date   Debit  Credit  Balance Transaction_Mode DR/CR_Indicator Transaction_ID  Recipient_Name  Bank    UPI_ID Note  Amount
0       2023-04-01     0.0  1500.0  1614.48              INB              CR   309111290781  PHONE_TRANSFER   NaN       NaN  Son  1500.0
1       2023-04-01    75.0     0.0  1539.48              UPI              DR   309122218462        ASIM HEM  HDFC  asimshah  U

In [3]:
date_col = bk.columns[0]
print(f"\nDate range: {bk[date_col].min()} to {bk[date_col].max()}")



Date range: 2023-04-01 to 2026-03-14


In [4]:
# ---- SMS Financial CSV ----
print("\n" + "=" * 80)
print("SMS FINANCIAL (true_financial_sms.csv)")
print("=" * 80)
sms = pd.read_csv("CSVS/true_financial_sms.csv")
print(f"Shape: {sms.shape}")
print(f"\nColumns: {list(sms.columns)}")
print(f"\nDtypes:\n{sms.dtypes}")
print(f"\nFirst 3 rows:")
print(sms.head(3).to_string())
print(f"\nNull counts:\n{sms.isnull().sum()}")


SMS FINANCIAL (true_financial_sms.csv)
Shape: (63, 15)

Columns: ['id', 'sender', 'body', 'timestamp_ms', 'timestamp_human', 'device_id', 'is_financial', 'amount', 'direction', 'bank', 'upi_id', 'recipient', 'date', 'ref_id', 'entity']

Dtypes:
id                  object
sender              object
body                object
timestamp_ms         int64
timestamp_human     object
device_id           object
is_financial          bool
amount             float64
direction           object
bank                object
upi_id             float64
recipient           object
date                object
ref_id             float64
entity              object
dtype: object

First 3 rows:
                                     id       sender                                                                                                                                                         body   timestamp_ms      timestamp_human         device_id  is_financial  amount direction bank  upi_id            

In [5]:
if "date" in sms.columns:
    print(f"\nDate range: {sms['date'].min()} to {sms['date'].max()}")



Date range: 2026-01-01 to 2026-05-08


In [6]:
# ---- Comparison ----
print("\n" + "=" * 80)
print("COMPARISON")
print("=" * 80)

# Direction distribution
print("\nBank Statement - DR/CR distribution:")
if "DR/CR_Indicator" in bk.columns:
    print(bk["DR/CR_Indicator"].value_counts())
elif "dr_cr_indicator" in bk.columns:
    print(bk["dr_cr_indicator"].value_counts())

print("\nSMS - Direction distribution:")
if "direction" in sms.columns:
    print(sms["direction"].value_counts())



COMPARISON

Bank Statement - DR/CR distribution:
DR/CR_Indicator
DR    1294
CR     359
Name: count, dtype: int64

SMS - Direction distribution:
direction
DEBIT     32
CREDIT    31
Name: count, dtype: int64


In [7]:
# Amount stats
print("\nBank Statement - Amount stats:")
for col in ["Amount", "Debit", "Credit"]:
    if col in bk.columns:
        print(f"  {col}: count={bk[col].notna().sum()}, sum={bk[col].sum():.2f}, min={bk[col].min()}, max={bk[col].max()}")

print("\nSMS - Amount stats:")
if "amount" in sms.columns:
    print(f"  amount: count={sms['amount'].notna().sum()}, sum={sms['amount'].sum():.2f}, min={sms['amount'].min()}, max={sms['amount'].max()}")



Bank Statement - Amount stats:
  Amount: count=1653, sum=1526.59, min=-150000.0, max=150000.0
  Debit: count=1653, sum=642299.95, min=0.0, max=150000.0
  Credit: count=1653, sum=643826.54, min=0.0, max=150000.0

SMS - Amount stats:
  amount: count=63, sum=344261.42, min=1.0, max=75000.0


In [8]:

# Transaction mode
print("\nBank Statement - Transaction modes:")
if "Transaction_Mode" in bk.columns:
    print(bk["Transaction_Mode"].value_counts())

print("\nSMS - Transaction modes (from sender/body):")
# Check if there's a mode column
for col in sms.columns:
    if "mode" in col.lower():
        print(sms[col].value_counts())

# Overlap analysis: matching by ref_id
print("\n" + "=" * 80)
print("OVERLAP ANALYSIS (matching by reference ID)")
print("=" * 80)

bk_ref_col = None
sms_ref_col = None
for c in bk.columns:
    if "reference" in c.lower() or "transaction_id" in c.lower():
        bk_ref_col = c
        break
for c in sms.columns:
    if "ref" in c.lower():
        sms_ref_col = c
        break

if bk_ref_col and sms_ref_col:
    bk_refs = set(bk[bk_ref_col].dropna().astype(str))
    sms_refs = set(sms[sms_ref_col].dropna().astype(str))
    overlap = bk_refs & sms_refs
    print(f"Bank statement refs: {len(bk_refs)}")
    print(f"SMS refs: {len(sms_refs)}")
    print(f"Overlap (in both): {len(overlap)}")
    print(f"Only in bank statement: {len(bk_refs - sms_refs)}")
    print(f"Only in SMS: {len(sms_refs - bk_refs)}")
else:
    print(f"Could not find ref columns. Bank: {bk_ref_col}, SMS: {sms_ref_col}")



Bank Statement - Transaction modes:
Transaction_Mode
UPI     1481
INB       86
IMP       41
NEFT       4
POS        3
ACH        2
Name: count, dtype: int64

SMS - Transaction modes (from sender/body):

OVERLAP ANALYSIS (matching by reference ID)
Bank statement refs: 1549
SMS refs: 42
Overlap (in both): 0
Only in bank statement: 1549
Only in SMS: 42


In [13]:
# 1. Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
import plotly.io as pio
pio.renderers.default = 'browser'
warnings.filterwarnings('ignore')

# 2. Load the Data
excel_path = r"CSVS/SpendWise2k26.xlsx"
sms_path = r"CSVS/true_financial_sms.csv"

df_excel = pd.read_excel(excel_path)
df_sms = pd.read_csv(sms_path)

# 3. Unify the Data Schemas
# We need to map both datasets to a common structure: Date, Amount, Type, Recipient, Mode

# Process Excel
df_excel_clean = pd.DataFrame({
    'Date': pd.to_datetime(df_excel['Transaction_Date']),
    'Amount': df_excel['Debit'].fillna(0) + df_excel['Credit'].fillna(0),
    'Type': df_excel['DR/CR_Indicator'].map({'DR': 'DEBIT', 'CR': 'CREDIT'}),
    'Recipient': df_excel['Recipient_Name'].fillna('Unknown'),
    'Mode': df_excel['Transaction_Mode'].fillna('OTHER'),
    'Bank': df_excel['Bank'].fillna('Unknown'),
    'Source': 'Bank Statement'
})

# Process SMS
df_sms_clean = pd.DataFrame({
    'Date': pd.to_datetime(df_sms['date']),
    'Amount': df_sms['amount'].fillna(0),
    'Type': df_sms['direction'].map({'DEBIT': 'DEBIT', 'CREDIT': 'CREDIT', 'DR': 'DEBIT', 'CR': 'CREDIT'}),
    'Recipient': df_sms['entity'].fillna(df_sms['recipient']).fillna('Unknown'),
    'Mode': 'UPI', # Defaulting to UPI for SMS if mode isn't explicitly in the CSV columns
    'Bank': df_sms['bank'].fillna('Unknown'),
    'Source': 'SMS'
})

# 4. Merge and Sort
df_unified = pd.concat([df_excel_clean, df_sms_clean], ignore_index=True)
df_unified = df_unified.sort_values('Date').reset_index(drop=True)

# Add helper columns for grouping
df_unified['Month_Year'] = df_unified['Date'].dt.to_period('M').astype(str)

print(f"Total Unified Transactions: {len(df_unified)}")

# ==========================================
# VISUALIZATION 1: Income vs Expenses by Month
# ==========================================
monthly_summary = df_unified.groupby(['Month_Year', 'Type'])['Amount'].sum().reset_index()

fig1 = px.bar(
    monthly_summary, x='Month_Year', y='Amount', color='Type',
    barmode='group', title='Monthly Income vs Expenses',
    color_discrete_map={'DEBIT': '#EF4444', 'CREDIT': '#10B981'},
    labels={'Amount': 'Amount (₹)', 'Month_Year': 'Month'}
)
fig1.update_layout(template='plotly_dark')
fig1.show()

# ==========================================
# VISUALIZATION 2: Daily Spending Trend (Last 90 Days of Data)
# ==========================================
debits_only = df_unified[df_unified['Type'] == 'DEBIT']
latest_date = debits_only['Date'].max()
last_90_days = debits_only[debits_only['Date'] >= (latest_date - pd.Timedelta(days=90))]

daily_spend = last_90_days.groupby('Date')['Amount'].sum().reset_index()

fig2 = px.line(
    daily_spend, x='Date', y='Amount', 
    title='Daily Spending Trend (Last 90 Days of Activity)',
    markers=True, line_shape='spline'
)
fig2.update_traces(line_color='#EF4444')
fig2.update_layout(template='plotly_dark')
fig2.show()

# ==========================================
# VISUALIZATION 3: Top 10 Recipients (Where is the money going?)
# ==========================================
# Exclude generic names if needed, like 'PHONE_TRANSFER'
top_recipients = debits_only[~debits_only['Recipient'].str.contains('PHONE_TRANSFER', case=False)]
top_recipients = top_recipients.groupby('Recipient')['Amount'].sum().reset_index()
top_recipients = top_recipients.sort_values('Amount', ascending=False).head(10)

fig3 = px.bar(
    top_recipients, x='Amount', y='Recipient', orientation='h',
    title='Top 10 Expense Categories / Recipients',
    color='Amount', color_continuous_scale='Reds'
)
fig3.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_dark')
fig3.show()

# ==========================================
# VISUALIZATION 4: Transaction Modes Breakdown
# ==========================================
mode_summary = df_unified['Mode'].value_counts().reset_index()
mode_summary.columns = ['Mode', 'Count']

fig4 = px.pie(
    mode_summary, names='Mode', values='Count', hole=0.4,
    title='Transaction Methods Used'
)
fig4.update_layout(template='plotly_dark')
fig4.show()


Total Unified Transactions: 1716
